# Download Semua Data Harga Saham (Fixed)
Notebook ini digunakan untuk mengunduh semua data harga historis saham yang terdaftar di `List-Perusahaan/Perusahaan.csv` dan menyimpannya ke dalam folder `Dataset/` secara otomatis.

**FIX PENTING:** Versi sebelumnya pakai `pd.concat` + `drop_duplicates(keep='last')` untuk gabung data lama & baru. Karena `saham` (download baru dari yfinance) TIDAK punya kolom broker summary, dan ditaruh setelah `old` di concat, `keep='last'` selalu memenangkan baris baru yang kolom brokernya NaN — jadi SEMUA data broker yang udah discrape ke-wipe tiap kali notebook ini dijalankan ulang.

Fix di sini: update kolom HARGA saja lewat index tanggal, kolom broker (dan kolom lain di luar harga) sama sekali gak disentuh.

In [1]:
import yfinance as yf
import pandas as pd
import os
import time

KOLOM_HARGA = ["Open", "High", "Low", "Close", "Volume"]

def historical_price(Code_Saham):
    path_csv = f"./Dataset/{Code_Saham}.csv"

    # Buat folder Dataset otomatis di root jika belum ada
    os.makedirs(os.path.dirname(path_csv), exist_ok=True)

    # Download data
    saham = yf.download(Code_Saham, period="5d", interval="1d", progress=False)
    if saham.empty:
        raise ValueError("Data kosong atau ticker tidak ditemukan di yfinance")

    # Fix multi-index header
    saham.columns = [col[0] if isinstance(col, tuple) else col for col in saham.columns]

    # Jadikan Date kolom biasa lalu langsung dijadikan index string 'YYYY-MM-DD'
    saham.reset_index(inplace=True)
    saham["Date"] = pd.to_datetime(saham["Date"]).dt.strftime("%Y-%m-%d")
    saham = saham.set_index("Date")

    kolom_harga_ada = [c for c in KOLOM_HARGA if c in saham.columns]

    if os.path.exists(path_csv):
        old = pd.read_csv(path_csv)
        old["Date"] = pd.to_datetime(old["Date"]).dt.strftime("%Y-%m-%d")
        old = old.set_index("Date")

        # 1) Update HANYA kolom harga untuk tanggal yang udah ada di old.
        #    Kolom broker & kolom lain di old SAMA SEKALI GAK DISENTUH.
        tanggal_overlap = old.index.intersection(saham.index)
        for col in kolom_harga_ada:
            old.loc[tanggal_overlap, col] = saham.loc[tanggal_overlap, col]

        # 2) Tambahin tanggal baru yang belum ada di old (row baru, kolom
        #    broker otomatis NaN karena emang belum pernah discrape)
        tanggal_baru = saham.index.difference(old.index)
        if len(tanggal_baru) > 0:
            df = pd.concat([old, saham.loc[tanggal_baru]])
        else:
            df = old

        df = df.sort_index()
    else:
        df = saham.sort_index()

    # Simpan kembali ke CSV, Date balik jadi kolom biasa
    df = df.reset_index()
    df.to_csv(path_csv, index=False)
    return len(df)


In [2]:
# 1. Load list perusahaan
list_perusahaan_path = "./List-Perusahaan/Perusahaan.csv"
if not os.path.exists(list_perusahaan_path):
    raise FileNotFoundError(f"File {list_perusahaan_path} tidak ditemukan! Pastikan Anda sudah menjalankan scraping list perusahaan terlebih dahulu.")

df_perusahaan = pd.read_csv(list_perusahaan_path)
tickers = df_perusahaan['Ticker_YF'].dropna().unique().tolist()
total_tickers = len(tickers)

print(f"Total saham yang akan didownload: {total_tickers}")


Total saham yang akan didownload: 826


In [3]:
# 2. Loop download dengan error handling
success_count = 0
failed_tickers = []

print("Memulai proses download...\n")

for i, ticker in enumerate(tickers, 1):
    try:
        # Beri jeda 0.5 detik agar tidak terkena rate limit dari API Yahoo Finance
        time.sleep(0.5)

        total_rows = historical_price(ticker)
        success_count += 1
        print(f"[{i}/{total_tickers}] {ticker}: BERHASIL ({total_rows} baris data)")

    except Exception as e:
        failed_tickers.append((ticker, str(e)))
        print(f"[{i}/{total_tickers}] {ticker}: GAGAL - {str(e)}")

print("\n=======================================")
print(f"Proses Selesai!")
print(f"Berhasil: {success_count}/{total_tickers}")
print(f"Gagal: {len(failed_tickers)}/{total_tickers}")
print("=======================================")

if failed_tickers:
    print("\nDaftar ticker yang gagal:")
    for t, err in failed_tickers:
        print(f"- {t}: {err}")


Memulai proses download...

[1/826] AMRT.JK: BERHASIL (10 baris data)
[2/826] MLPL.JK: BERHASIL (10 baris data)
[3/826] TAPG.JK: BERHASIL (10 baris data)
[4/826] MPPA.JK: BERHASIL (10 baris data)
[5/826] MIDI.JK: BERHASIL (10 baris data)
[6/826] LAPD.JK: BERHASIL (10 baris data)
[7/826] RANC.JK: BERHASIL (10 baris data)
[8/826] DMND.JK: BERHASIL (10 baris data)
[9/826] HERO.JK: BERHASIL (10 baris data)
[10/826] PCAR.JK: BERHASIL (10 baris data)
[11/826] EPMT.JK: BERHASIL (10 baris data)
[12/826] AMMS.JK: BERHASIL (10 baris data)
[13/826] SDPC.JK: BERHASIL (10 baris data)
[14/826] KMDS.JK: BERHASIL (10 baris data)
[15/826] BUAH.JK: BERHASIL (10 baris data)
[16/826] DAYA.JK: BERHASIL (10 baris data)
[17/826] WICO.JK: BERHASIL (10 baris data)


$MAMIP.JK: possibly delisted; no price data found  (period=5d)

1 Failed download:
['MAMIP.JK']: possibly delisted; no price data found  (period=5d)


[18/826] MAMIP.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: GWAA.JK"}}}
$GWAA.JK: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['GWAA.JK']: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


[19/826] GWAA.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[20/826] INDF.JK: BERHASIL (10 baris data)
[21/826] ICBP.JK: BERHASIL (10 baris data)
[22/826] AALI.JK: BERHASIL (10 baris data)
[23/826] JPFA.JK: BERHASIL (10 baris data)
[24/826] MYOR.JK: BERHASIL (10 baris data)
[25/826] CPIN.JK: BERHASIL (10 baris data)
[26/826] LSIP.JK: BERHASIL (10 baris data)
[27/826] RLCO.JK: BERHASIL (10 baris data)
[28/826] GZCO.JK: BERHASIL (10 baris data)
[29/826] FORE.JK: BERHASIL (10 baris data)
[30/826] BWPT.JK: BERHASIL (10 baris data)
[31/826] JARR.JK: BERHASIL (10 baris data)
[32/826] SIMP.JK: BERHASIL (10 baris data)
[33/826] ULTJ.JK: BERHASIL (10 baris data)
[34/826] ASHA.JK: BERHASIL (10 baris data)
[35/826] COCO.JK: BERHASIL (10 baris data)
[36/826] WMUU.JK: BERHASIL (10 baris data)
[37/826] CPRO.JK: BERHASIL (10 baris data)
[38/826] IKAN.JK: BERHASIL (10 baris data)
[39/826] DSNG.JK: BERHASIL (10 baris data)
[40/826] CLEO.JK: BERHASIL (10 baris data)
[41/826] OILS.JK: B

...

[741/826] NPII.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[742/826] ASII.JK: BERHASIL (10 baris data)
[743/826] BNBR.JK: BERHASIL (10 baris data)
[744/826] BHIT.JK: BERHASIL (10 baris data)
[745/826] FOLK.JK: BERHASIL (10 baris data)
[746/826] ZBRA.JK: BERHASIL (10 baris data)
[747/826] KING.JK: BERHASIL (10 baris data)
[748/826] UNTR.JK: BERHASIL (10 baris data)
[749/826] PIPA.JK: BERHASIL (10 baris data)
[750/826] IMPC.JK: BERHASIL (10 baris data)
[751/826] HEXA.JK: BERHASIL (10 baris data)
[752/826] LABA.JK: BERHASIL (10 baris data)
[753/826] NTBK.JK: BERHASIL (10 baris data)
[754/826] SMIL.JK: BERHASIL (10 baris data)
[755/826] KUAS.JK: BERHASIL (10 baris data)
[756/826] SINI.JK: BERHASIL (10 baris data)
[757/826] PTMP.JK: BERHASIL (10 baris data)
[758/826] TOTO.JK: BERHASIL (10 baris data)
[759/826] MARK.JK: BERHASIL (10 baris data)
[760/826] HOPE.JK: BERHASIL (10 baris data)
[761/826] KOBX.JK: BERHASIL (10 baris data)
[762/826] CTTH.JK: BERHASIL (10 baris dat

$KRAH.JK: possibly delisted; no price data found  (period=5d)

1 Failed download:
['KRAH.JK']: possibly delisted; no price data found  (period=5d)


[785/826] KRAH.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance


$ASIA.JK: possibly delisted; no price data found  (period=5d)

1 Failed download:
['ASIA.JK']: possibly delisted; no price data found  (period=5d)


[786/826] ASIA.JK: GAGAL - Data kosong atau ticker tidak ditemukan di yfinance
[787/826] GIAA.JK: BERHASIL (10 baris data)
[788/826] BIRD.JK: BERHASIL (10 baris data)
[789/826] ASSA.JK: BERHASIL (10 baris data)
[790/826] IMJS.JK: BERHASIL (10 baris data)
[791/826] WEHA.JK: BERHASIL (10 baris data)
[792/826] CMPP.JK: BERHASIL (10 baris data)
[793/826] TAXI.JK: BERHASIL (10 baris data)
[794/826] TRJA.JK: BERHASIL (10 baris data)
[795/826] LRNA.JK: BERHASIL (10 baris data)
[796/826] BPTR.JK: BERHASIL (10 baris data)
[797/826] SAFE.JK: BERHASIL (10 baris data)
[798/826] HELI.JK: BERHASIL (10 baris data)
[799/826] WBSA.JK: BERHASIL (10 baris data)
[800/826] PJHB.JK: BERHASIL (10 baris data)
[801/826] SMDR.JK: BERHASIL (10 baris data)
[802/826] LAJU.JK: BERHASIL (10 baris data)
[803/826] BLOG.JK: BERHASIL (10 baris data)
[804/826] TMAS.JK: BERHASIL (10 baris data)
[805/826] SDMU.JK: BERHASIL (10 baris data)
[806/826] JAYA.JK: BERHASIL (10 baris data)
[807/826] KLAS.JK: BERHASIL (10 baris dat